# Cell-defined Routes

A cell can serve HTTP requests from its kernel's Runtime. In this notebook, the global `app` is a [Hono](https://hono.dev) app, and `claimPrefix(prefix)` claims a URL prefix for the kernel. Requests under a claimed prefix reach the handlers registered on `app` while the kernel's Runtime is running.

Run the cells in order from top to bottom.

## Claim a prefix

`claimPrefix(prefix)` claims a URL prefix for this kernel. Prefixes match whole path segments.

Prefixes are compared with the URL path from the site origin. If JupyterLite is served from a subpath, such as `/my-site/`, include that subpath in the prefix and in the `fetch` URLs below.

In [1]:
claimPrefix('/demo/')

Claimed prefixes: /demo/


## Ask for the claimed prefixes

`claimedPrefixes()` returns a Promise that resolves to the prefixes this kernel has claimed.

In [2]:
await claimedPrefixes()

['/demo/']

## Add middleware

Middleware registered with `app.use` runs for matching requests. Register it before the routes it should cover, because routes registered earlier can finish the request before the middleware runs.

This middleware adds a header to every response under `/demo/`.

In [3]:
app.use('/demo/*', async (c, next) => {
  await next();
  c.header('x-demo', 'cell-routes');
})

bound Hono { get: [Function], post: [Function], put: [Function], delete: [Function], options: [Function], ... }

## Register a JSON route

Routes are registered on `app` with the [Hono](https://hono.dev) API. This route returns a JSON response.

In [4]:
app.get('/demo/hello', c => c.json({ message: 'hello from a notebook cell' }))

bound Hono { get: [Function], post: [Function], put: [Function], delete: [Function], options: [Function], ... }

## Send a request to the route

Cells can call their own routes with `fetch`. The request is routed to this kernel's Runtime, so the response comes from the handler above.

In [5]:
const response = await fetch('/demo/hello');
await response.json()

{ message: 'hello from a noteboo...' }

## Serve a plain-text root response

Use `c.text()` for a plain-text response. The route path is rooted at `/demo/`, inside the prefix claimed above.

In [6]:
app.get('/demo/', c => c.text('Hello from the notebook Runtime'));

In [7]:
const greeting = await fetch('/demo/');
({ status: greeting.status, body: await greeting.text() });

## Path parameters

Hono routes can capture path segments with `:name`. The handler reads them with `c.req.param()`.

In [8]:
app.get('/demo/greet/:name', c => c.text(`Hello, ${c.req.param('name')}!`))

bound Hono { get: [Function], post: [Function], put: [Function], delete: [Function], options: [Function], ... }

In [9]:
await (await fetch('/demo/greet/Ada')).text()

'Hello, Ada!'

## Read a JSON request body

Handlers can read the request body. This route echoes the JSON it receives and responds with status `201`.

In [10]:
app.post('/demo/echo', async c => c.json({ received: await c.req.json() }, 201))

bound Hono { get: [Function], post: [Function], put: [Function], delete: [Function], options: [Function], ... }

In [11]:
const created = await fetch('/demo/echo', {
  method: 'POST',
  headers: { 'content-type': 'application/json' },
  body: JSON.stringify({ name: 'Ada', language: 'JavaScript' })
});
await created.json() 

{ received: {...} }

## Errors

- A path under a claimed prefix that no route matches returns `404`.
- A handler that throws returns `500`. The error is also shown in the notebook as an error output.

In [12]:
(await fetch('/demo/missing')).status

404

In [13]:
app.get('/demo/boom', () => {
  throw new Error('boom from a route handler');
})

bound Hono { get: [Function], post: [Function], put: [Function], delete: [Function], options: [Function], ... }

In [14]:
(await fetch('/demo/boom')).status

500

## Re-running cells

Re-running a cell replaces the routes that cell registered. Edit the handler for `/demo/hello`, run its cell again, and send the request again to see the new response.

Restarting the kernel releases its claims and clears its routes. Run **Claim a prefix** again after a restart.